In [0]:
%python

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.crypto_silver
""")

In [0]:
%python
from pyspark.sql import functions as F

# Read data from Bronze
bronze_df = spark.table(
    "workspace.crypto_bronze.market_candles_v2"
)

# Transform data types
silver_df = (
    bronze_df
    .withColumn(
        "open_time",
        F.expr("timestamp_millis(open_time_ms)")
    )
    .withColumn(
        "close_time",
        F.expr("timestamp_millis(close_time_ms)")
    )
    .withColumn("open", F.col("open").cast("decimal(20,8)"))
    .withColumn("high", F.col("high").cast("decimal(20,8)"))
    .withColumn("low", F.col("low").cast("decimal(20,8)"))
    .withColumn("close", F.col("close").cast("decimal(20,8)"))
    .withColumn("volume", F.col("volume").cast("decimal(28,8)"))
)

display(
    silver_df.select(
        "symbol",
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ).limit(10)
)

silver_df.printSchema()

In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Order candles chronologically within each trading pair
window = Window.partitionBy(
    "symbol", "interval"
).orderBy("open_time")

# Calculate time difference between consecutive candles
checked_df = (
    silver_df
    .withColumn(
        "previous_time",
        F.lag("open_time").over(window)
    )
    .withColumn(
        "gap_seconds",
        F.unix_timestamp("open_time") -
        F.unix_timestamp("previous_time")
    )
)

# Check for unexpected intervals
invalid_intervals = checked_df.filter(
    F.col("previous_time").isNotNull() &
    (F.col("gap_seconds") != 3600)
).count()

print("Invalid hourly intervals:", invalid_intervals)

assert invalid_intervals == 0, "Missing or irregular candles detected"

print("Time series validation passed!")


In [0]:
%python

# Keep only fully closed candles
closed_df = silver_df.filter(
    F.col("close_time") < F.current_timestamp()
)

print("All candles:", silver_df.count())
print("Closed candles:", closed_df.count())
print(
    "Excluded open candles:",
    silver_df.count() - closed_df.count()
)


In [0]:
TARGET_SILVER = "workspace.crypto_silver.market_candles"

(
    closed_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_SILVER)
)

print("Silver table saved!")


In [0]:
%sql
SELECT
    symbol,
    COUNT(*) AS total_candles
FROM workspace.crypto_silver.market_candles
GROUP BY symbol;